In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
sns.set_theme(style="whitegrid")
from tqdm.auto import tqdm
from ema_pytorch import EMA
from glob import glob
from data.build_dataloader import build_dataloader
import yaml
import json
import os
from scipy import signal

from utils.io_utils import seed_everything, load_FM_config, load_model
from utils.priors import PriorGenerator
from utils.evaluation import evaluate_FM, evaluate_SPFM
from momentfm import MOMENTPipeline
from models.SPFM import SPFM_Ensemble
from models.transformer_1d import Transformer1DModel as Transformer
from models.mamba_1d import Mamba1DModel as Mamba
from models.mamformer_1d import MambaTransformer1DModel as Mamformer

def parse_loss_type(path):
    """Extract the loss type from a results-folder slug, e.g. '.../l1_white' -> 'l1'."""
    if 'fw' in path:
        return 'FW'
    else:
        return path.split('/')[-1].split('_')[0]
    
def parse_loss_types(path):
    """Extract the list of loss types from a results-folder slug, e.g. '.../l1_l2_white' -> ['l1', 'l2']."""
    slug_parts = path.split('/')[-1].split('_')
    return slug_parts[:-1]

In [ ]:
""" FM evaluation """

seed_everything(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Structure: ['path', 'model', 'backbone', 'num_params']   
freq_models = [
    {
        'dataset': 'ISRUC',
        'path': './experiments/ISRUC/FM/Transformer/l2_white',
        'model': 'SPFM',
        'backbone': 'Transformer',
        'num_params': '6.01M'
    },
    {
        'dataset': 'Sleep-EDF',
        'path': './experiments/Sleep-EDF/FM/Uniform/Transformer/l2_white',
        'model': 'SPFM',
        'backbone': 'Transformer',
        'num_params': '6.01M'
    },
]

# Load MOMENT foundational TS model for FID calculation
moment_model = MOMENTPipeline.from_pretrained(
    '../foundational_models/moment_large_embedding',
    model_kwargs={'task_name': 'embedding'},
    torch_dtype=torch.float32,
    local_files_only=True
)
moment_model.init()
moment_model.to(device)
moment_model.eval()
print(f"MOMENT model loaded successfully on {device}")

columns = ['dataset', 'path', 'model', 'backbone', 'num_params', 'loss', 'psd', 'sinkhorn', 'ilse', 'swd', 'fid']
results_df = pd.DataFrame(freq_models, columns=columns)
results_df['loss'] = results_df['path'].apply(parse_loss_type)

# Synthetic Data from SoA Models Evaluation
num_runs = 5
all_metrics = {
    index: {'step': 0, 'sinkhorn': [], 'ilse': [], 'fid': [], 'swd': []}
    for index, _ in results_df.iterrows()
}
test_dl_cache = {}
for run_id in range(num_runs):
    seed_everything(123+run_id)
    print(f"Starting evaluation run {run_id + 1}/{num_runs}")
    for index, row in results_df.iterrows():
        with open(f"{row['path']}/configs/config.yaml", 'r') as file:
            config = yaml.safe_load(file)
        if row['dataset'] not in test_dl_cache:
            test_dl_cache[row['dataset']] = build_dataloader(config, 'test')
        test_dl = test_dl_cache[row['dataset']]

        loss = row['path'].split('/')[-1].split('_')[0]
        model = load_model(config['model'], loss_type=loss).to(device)
        ema = EMA(model, beta=0.995, update_every=10).to(device)
        
        if os.path.exists(f"{row['path']}/ckpt-100000.pt"):
            path = f"{row['path']}/ckpt-100000.pt"
            ckpt = torch.load(path)
            biggest_step = '100000'
        else:
            biggest_step = 0
            for ckpt_path in glob(f"{row['path']}/ckpt-*.pt"):
                ckpt_data = torch.load(ckpt_path)
                step = ckpt_data['step']
                if step > biggest_step:
                    biggest_step = step
                    ckpt, path = ckpt_data, ckpt_path
                
            print(f"Loading data from step {biggest_step}, checkpoint {os.path.basename(path)}")
        model.load_state_dict(ckpt['model'])
        ema.load_state_dict(ckpt['ema'])
        
        prior = PriorGenerator("white", None)

        metrics = evaluate_FM(model=ema.ema_model,
                              dataloader=test_dl['dataloader'],
                              prior=prior,
                              device=device,
                              moment_model=moment_model,
                              sampling_timesteps=50
                             )
        if run_id == 0:
            all_metrics['step'] = biggest_step        
        all_metrics[index]['sinkhorn'].append(metrics['sinkhorn'])
        all_metrics[index]['ilse'].append(metrics['ilse'].item())
        all_metrics[index]['fid'].append(metrics['fid'].item())
        all_metrics[index]['swd'].append(metrics['swd'].item())
        
        del model, ema, test_dl

print("Final Summary Metrics (mean ± std)")

for index, row in results_df.iterrows():
    metrics_run = all_metrics[index]
    # Calculate means and stds
    m_sinkhorn, s_sinkhorn = np.mean(metrics_run['sinkhorn']), np.std(metrics_run['sinkhorn'])
    m_ilse, s_ilse         = np.mean(metrics_run['ilse']), np.std(metrics_run['ilse'])
    m_fid, s_fid           = np.mean(metrics_run['fid']), np.std(metrics_run['fid'])
    m_swd, s_swd           = np.mean(metrics_run['swd']), np.std(metrics_run['swd'])
    
    # Format strings to insert directly back into the dataframe
    results_df.at[index, 'sinkhorn'] = f"{m_sinkhorn:.5f} ± {s_sinkhorn:.3f}"
    results_df.at[index, 'ilse'] = f"{m_ilse:.5f} ± {s_ilse:.3f}"
    results_df.at[index, 'fid'] = f"{m_fid:.5f} ± {s_fid:.3f}"
    results_df.at[index, 'swd'] = f"{m_swd:.5f} ± {s_swd:.3f}"

    print(f"[{row['dataset']}] {row['model']}, {row['loss']} | Sinkhorn: {m_sinkhorn:.4f} | ILSE: {m_ilse:.4f} | SWD: {m_swd:.4f} | FID: {m_fid:.4f}")
    
display(results_df.drop(columns=['path', 'psd']))

In [ ]:
""" Unified SPFM Evaluation """

seed_everything(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

best_models = [
    {
        'dataset': 'PTBXL',
        'path': './experiments/PTBXL/Unified_SPFM/Uniform/Transformer/l1_l2_white',
        'model': 'Unified SPFM',
        'backbone': 'Transformer',
        'num_params': '6.74M'
    },
    {
        'dataset': 'Chapman',
        'path': './experiments/Chapman/Unified_SPFM/Uniform/Transformer/l1_l2_white',
        'model': 'Unified SPFM',
        'backbone': 'Transformer',
        'num_params': '6.74M'
    },
]


# Load MOMENT foundational TS model for FID calculation
moment_model = MOMENTPipeline.from_pretrained(
    '../foundational_models/moment_large_embedding',
    model_kwargs={'task_name': 'embedding'},
    torch_dtype=torch.float32,
    local_files_only=True
)
moment_model.init()
moment_model.to(device)
moment_model.eval()
print(f"MOMENT model loaded successfully on {device}")

columns = ['dataset', 'path', 'model', 'backbone', 'num_params', 'sinkhorn', 'ilse', 'swd', 'fid']
results_df = pd.DataFrame(best_models, columns=columns)

# Synthetic Data from SoA Models Evaluation
num_runs = 5
all_metrics = {
    index: {'sinkhorn': [], 'ilse': [], 'fid': [], 'swd': []}
    for index, _ in results_df.iterrows()
}
bb_classes = {'Transformer': Transformer, 'Mamba': Mamba, 'Mamformer': Mamformer}

for run_id in range(num_runs):
    seed_everything(123+run_id)
    print(f"Starting evaluation run {run_id + 1}/{num_runs}")
    for index, row in results_df.iterrows():
        with open(f"{row['path']}/configs/config.yaml", 'r') as file:
            config = yaml.safe_load(file)
        test_dl = build_dataloader(config, 'test')
        dataset = test_dl
        lf_loss, hf_loss = row['path'].split('/')[-1].split('_')[-3], row['path'].split('/')[-1].split('_')[-2]
        model = SPFM_Ensemble(
            K=2,
            backbone_class=bb_classes[row['backbone']],
            means=torch.zeros(2, 12, device=device),
            stds=torch.zeros(2, 12, device=device),
            solver='euler',
            loss_types=[lf_loss, hf_loss],
            **config['model']['backbone']['params']
        ).to(device)
            
        ema = EMA(model, beta=0.995, update_every=10).to(device)
        
        if os.path.exists(f"{row['path']}/ckpt-100000.pt"):
            path = f"{row['path']}/ckpt-100000.pt"
            ckpt = torch.load(path)
        else:
            biggest_step = 0
            for ckpt_path in glob(f"{row['path']}/ckpt-*.pt"):
                ckpt_data = torch.load(ckpt_path)
                step = ckpt_data['step']
                if step > biggest_step:
                    biggest_step = step
                    ckpt, path = ckpt_data, ckpt_path

        model.load_state_dict(ckpt['model'])
        ema.load_state_dict(ckpt['ema'])
        
        prior = PriorGenerator("white", None)

        metrics = evaluate_SPFM(model=ema.ema_model,
                              dataloader=test_dl['dataloader'],
                              prior=prior,
                              device=device,
                              moment_model=moment_model,
                              sampling_timesteps=[20, 30],
                              solver='euler'
                             )
        
        all_metrics[index]['sinkhorn'].append(metrics['sinkhorn'])
        all_metrics[index]['ilse'].append(metrics['ilse'].item())
        all_metrics[index]['fid'].append(metrics['fid'].item())
        all_metrics[index]['swd'].append(metrics['swd'].item())
        
        del model, ema, test_dl

print("Final Summary Metrics (mean ± std)")

for index, row in results_df.iterrows():
    metrics_run = all_metrics[index]
    # Calculate means and stds
    m_sinkhorn, s_sinkhorn = np.mean(metrics_run['sinkhorn']), np.std(metrics_run['sinkhorn'])
    m_ilse, s_ilse         = np.mean(metrics_run['ilse']), np.std(metrics_run['ilse'])
    m_fid, s_fid           = np.mean(metrics_run['fid']), np.std(metrics_run['fid'])
    m_swd, s_swd           = np.mean(metrics_run['swd']), np.std(metrics_run['swd'])
    
    # Format strings to insert directly back into the dataframe
    results_df.at[index, 'sinkhorn'] = f"{m_sinkhorn:.5f} ± {s_sinkhorn:.3f}"
    results_df.at[index, 'ilse'] = f"{m_ilse:.5f} ± {s_ilse:.3f}"
    results_df.at[index, 'fid'] = f"{m_fid:.5f} ± {s_fid:.3f}"
    results_df.at[index, 'swd'] = f"{m_swd:.5f} ± {s_swd:.3f}"

    print(f"[{row['dataset']}] {row['model']} | Sinkhorn: {m_sinkhorn:.4f} | ILSE: {m_ilse:.4f} | SWD: {m_swd:.4f} | FID: {m_fid:.4f}")
    
display(results_df.drop(columns=['path']))

In [ ]:
""" SPFM Evaluation """
seed_everything(123)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

best_models = [
    {
        'dataset': 'ISRUC',
        'path': './experiments/ISRUC/SPFM/Uniform/Transformer/l1_l2_white',
        'model': 'SPFM',
        'backbone': 'Transformer',
        'num_params': '6.73M'
    },    
    {
        'dataset': 'Sleep-EDF',
        'path': './experiments/Sleep-EDF/SPFM/Uniform/Mamba/l1_l2_white',
        'model': 'SPFM',
        'backbone': 'Mamba',
        'num_params': '6.94M'
    } 
]

# Load MOMENT foundational TS model for FID calculation
moment_model = MOMENTPipeline.from_pretrained(
    '../foundational_models/moment_large_embedding',
    model_kwargs={'task_name': 'embedding'},
    torch_dtype=torch.float32,
    local_files_only=True
)
moment_model.init()
moment_model.to(device)
moment_model.eval()
print(f"MOMENT model loaded successfully on {device}")

columns = ['dataset', 'path', 'model', 'backbone', 'num_params', 'losses', 'psd', 'sinkhorn', 'ilse', 'swd', 'fid']
results_df = pd.DataFrame(best_models, columns=columns)
results_df['losses'] = results_df['path'].apply(parse_loss_types)

# Synthetic Data from SoA Models Evaluation
num_runs, K, C = 3, 2, 2
all_metrics = {
    index: {'sinkhorn': [], 'ilse': [], 'fid': [], 'swd': []}
    for index, _ in results_df.iterrows()
}
bb_classes = {'Transformer': Transformer, 'Mamba': Mamba, 'Mamformer': Mamformer}
test_dl_cache = {}

for run_id in range(num_runs):
    seed_everything(123+run_id)
    print(f"Starting evaluation run {run_id + 1}/{num_runs}")
    for index, row in results_df.iterrows():
        with open(f"{row['path']}/configs/config.yaml", 'r') as file:
            config = yaml.safe_load(file)
        if row['dataset'] not in test_dl_cache:
            test_dl_cache[row['dataset']] = build_dataloader(config, 'test')
        test_dl = test_dl_cache[row['dataset']]

        losses = row['path'].split('/')[-1].split('_')[:-1]
        model = SPFM_Ensemble(
            K=K,
            backbone_class=bb_classes[row['backbone']],
            means=torch.zeros(K, C, device=device),
            stds=torch.zeros(K, C, device=device),
            solver='euler',
            loss_types=losses,
            **config['model']['backbone']['params']
        ).to(device)

        emas = [EMA(model.models[k], beta=0.995, update_every=10).to(device) for k in range(K)]
        ckpt_path = f"{row['path']}/ckpt-100000.pt"
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt['model'], strict=False)
        prior = PriorGenerator("white", None)
        
        for k in range(K):
            emas[k].load_state_dict(ckpt["ema"][k])
            emas[k].to(device).eval()
            model.models[k] = emas[k]
            
        metrics = evaluate_SPFM(model=model,
                              dataloader=test_dl['dataloader'],
                              prior=prior,
                              device=device,
                              moment_model=moment_model,
                              sampling_timesteps=[20, 30],
                              solver='euler'
                             )
        
        all_metrics[index]['sinkhorn'].append(metrics['sinkhorn'])
        all_metrics[index]['ilse'].append(metrics['ilse'].item())
        all_metrics[index]['fid'].append(metrics['fid'].item())
        all_metrics[index]['swd'].append(metrics['swd'].item())
        
        del model, emas, test_dl

In [ ]:
print("Final Summary Metrics (mean ± std)")

for index, row in results_df.iterrows():
    metrics_run = all_metrics[index]
    # Calculate means and stds
    m_sinkhorn, s_sinkhorn = np.mean(metrics_run['sinkhorn']), np.std(metrics_run['sinkhorn'])
    m_ilse, s_ilse         = np.mean(metrics_run['ilse']), np.std(metrics_run['ilse'])
    m_fid, s_fid           = np.mean(metrics_run['fid']), np.std(metrics_run['fid'])
    m_swd, s_swd           = np.mean(metrics_run['swd']), np.std(metrics_run['swd'])
    
    # Format strings to insert directly back into the dataframe
    results_df.at[index, 'sinkhorn'] = f"{m_sinkhorn:.5f} ± {s_sinkhorn:.3f}"
    results_df.at[index, 'ilse'] = f"{m_ilse:.5f} ± {s_ilse:.3f}"
    results_df.at[index, 'fid'] = f"{m_fid:.5f} ± {s_fid:.3f}"
    results_df.at[index, 'swd'] = f"{m_swd:.5f} ± {s_swd:.3f}"

    print(f"[{row['dataset']}] {row['model']} | Sinkhorn: {m_sinkhorn:.4f} | ILSE: {m_ilse:.4f} | SWD: {m_swd:.4f} | FID: {m_fid:.4f}")
    
display(results_df.drop(columns=['path', 'psd']))